# no3 Colab setup: persistent Drive output + temporary local data

This is the protocol-locked Google Colab derivative of canonical `no3`:
Standard 3D U-Net, outer fold 3, training seed 2026. It may count
as run `no3` only if every dependency, split, CUDA, locked 96³-memory,
training, prediction, and archive assertion passes. It never lowers the patch
size or changes the experiment for a weaker GPU.

Before using **Run all**:

1. Put the extracted `MU-Glioma-Post` folder at
   `/content/drive/MyDrive/MU-Glioma-Post`, or put
   `MU-Glioma-Post-Kaggle.bundle` in `MyDrive`.
2. In Colab choose **Runtime → Change runtime type → GPU**.
3. Use **Run all** and authorize the Google Drive mount.
4. If the dependency cell installs the frozen TensorFlow/Keras versions and
   asks for a restart, choose **Runtime → Restart session**, then use
   **Run all** once more.

The dataset is verified and staged from Drive to Colab's faster `/content`
disk. Checkpoints, logs, predictions, and the final ZIP remain under
`MyDrive/MU_Glioma_35_Run_Outputs`, so a runtime interruption can resume.

Only the three path settings in the environment cell may be changed for a
different Drive layout. Do not change the run constants, 96³ patch, fold,
seed, steps, stopping rule, model, or metrics.


## Prespecified experiment contract

- Run ID: `no3`
- Architecture: Standard 3D U-Net
- Cross-validation fold: 3 of 5
- Split seed: 2026; training seed: 2026
- Fit training: 381 scans / 130 patients
- Inner tuning: 94 scans / 32 patients
- Untouched outer test: 119 scans / 41 patients
- Split SHA-256: `03e6710ea8a4eb8a4e679830cf1d64bde85bd9c31bfc502cf292b9eb6079023e`
- Maximum 80 epochs; best weights restored
- Early stopping monitors inner-tuning foreground Dice after epoch 20 and stops
  after eight epochs without a meaningful 0.002 improvement.

The patch pipeline, optimizer, loss, and full-volume evaluation are held fixed
across Keras architectures. The outer test fold is not used by training,
learning-rate scheduling, early stopping, or checkpoint selection. Every ZIP
includes outer-test prediction volumes, the best
model, complete overlap/surface/lesion metrics, absent-reference uncertainty,
efficiency telemetry, checksums, logs, split, configuration, and environment
provenance. Do not edit the fold, seed, metric definitions, or stopping rule.


## 1. Colab environment, reproducibility, Drive, and persistent output


In [ ]:
MODEL_DISPLAY_NAME = 'Adaptive Standard 3D U-Net'
ARCHITECTURE_ID = 'adaptive_standard_3d_unet'
OUTPUT_NAME = 'MU_Glioma_no3_unet_fold3_seed2026'
MODEL_FILE = 'no3_unet_fold3_seed2026_best.keras'
# Colab setup: select a GPU runtime, mount Drive, then Run all.
import hashlib
import importlib.util
import importlib.metadata
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

EXPECTED_TENSORFLOW_VERSION = "2.20.0"
EXPECTED_KERAS_VERSION = "3.13.2"


def installed_distribution_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


environment_pins = {
    "tensorflow": EXPECTED_TENSORFLOW_VERSION,
    "keras": EXPECTED_KERAS_VERSION,
}
pin_mismatches = {
    name: (installed_distribution_version(name), expected)
    for name, expected in environment_pins.items()
    if installed_distribution_version(name) != expected
}
if pin_mismatches:
    print("Installing frozen Colab GPU environment:", pin_mismatches)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            f"tensorflow[and-cuda]=={EXPECTED_TENSORFLOW_VERSION}",
            f"keras=={EXPECTED_KERAS_VERSION}",
        ],
        check=True,
    )
    raise RuntimeError(
        "Frozen TensorFlow/Keras versions were installed successfully. Choose "
        "Runtime > Restart session, then use Run all again before training."
    )

if importlib.util.find_spec("nibabel") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "nibabel"],
        check=True,
    )

if importlib.util.find_spec("surface_distance") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "surface-distance==0.1"],
        check=True,
    )

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from tensorflow import keras
from tensorflow.keras import layers

assert tf.__version__ == EXPECTED_TENSORFLOW_VERSION, (
    f"Expected TensorFlow {EXPECTED_TENSORFLOW_VERSION}, found {tf.__version__}. "
    "Restart the Colab session after the pinned installation cell."
)
assert keras.__version__ == EXPECTED_KERAS_VERSION, (
    f"Expected Keras {EXPECTED_KERAS_VERSION}, found {keras.__version__}. "
    "Restart the Colab session after the pinned installation cell."
)

gpu_devices = tf.config.list_physical_devices("GPU")
assert gpu_devices, (
    "No CUDA GPU is visible. In Colab choose Runtime > Change runtime type > GPU, "
    "reconnect, and use Run all again."
)
for gpu_device in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError as error:
        raise RuntimeError(
            "TensorFlow initialized the GPU before memory growth was configured. "
            "Restart the Colab session and use Run all."
        ) from error

gpu_details = [tf.config.experimental.get_device_details(device) for device in gpu_devices]
try:
    nvidia_smi_record = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception as error:
    raise RuntimeError("Colab assigned a GPU, but nvidia-smi is unusable.") from error

RUN_ID = 'no3'
SPLIT_SEED = 2026
INNER_SPLIT_SEED = 6205
TRAINING_SEED = 2026
CV_FOLD = 3
SEED = TRAINING_SEED
EXPECTED_FIT_TRAIN_CASES = 381
EXPECTED_FIT_TRAIN_PATIENTS = 130
EXPECTED_TUNING_CASES = 94
EXPECTED_TUNING_PATIENTS = 32
EXPECTED_OUTER_TEST_CASES = 119
EXPECTED_OUTER_TEST_PATIENTS = 41
VAL_FRACTION = 0.20
NUM_CLASSES = 5
PATCH_SIZE = (96, 96, 96)
PATCHES_PER_CASE = 4
BATCH_SIZE = 1
EPOCHS = 80
MIN_IMPROVEMENT = 0.002
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_START_EPOCH = 20
STEPS_PER_EPOCH = 256
VALIDATION_STEPS = 64
LEARNING_RATE = 2e-4
INFERENCE_STRIDE = (64, 64, 64)
SAVE_PREDICTIONS = True
EXPECTED_SPLIT_SHA256 = "03e6710ea8a4eb8a4e679830cf1d64bde85bd9c31bfc502cf292b9eb6079023e"

COLAB_DRIVE_ROOT = Path(
    os.environ.get("MU_GLIOMA_DRIVE_ROOT", "/content/drive/MyDrive")
).expanduser().resolve()
COLAB_DATASET_SOURCE = Path(
    os.environ.get(
        "MU_GLIOMA_DATASET_SOURCE",
        COLAB_DRIVE_ROOT / "MU-Glioma-Post",
    )
).expanduser().resolve()
COLAB_OUTPUT_ROOT = Path(
    os.environ.get(
        "MU_GLIOMA_OUTPUT_ROOT",
        COLAB_DRIVE_ROOT / "MU_Glioma_35_Run_Outputs",
    )
).expanduser().resolve()
COLAB_SCRATCH_ROOT = Path("/content/mu_glioma_no3_scratch")
COLAB_STAGE_DATA_LOCALLY = True

assert COLAB_DRIVE_ROOT.is_dir(), f"Google Drive mount missing: {COLAB_DRIVE_ROOT}"
COLAB_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT = COLAB_OUTPUT_ROOT / OUTPUT_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.mixed_precision.set_global_policy("mixed_float16")

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", gpu_devices)
print("GPU details:", gpu_details)
print("nvidia-smi:", nvidia_smi_record)
print("Drive root:", COLAB_DRIVE_ROOT)
print("Dataset source:", COLAB_DATASET_SOURCE)
print("Mixed precision:", keras.mixed_precision.global_policy())
print("Output:", OUTPUT)

environment_record = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "nibabel": nib.__version__,
    "surface_distance": importlib.metadata.version("surface-distance"),
    "gpu_devices": [str(device) for device in gpu_devices],
    "gpu_details": gpu_details,
    "nvidia_smi": nvidia_smi_record,
    "execution_platform": "google_colab_cuda",
    "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG"),
    "drive_root": str(COLAB_DRIVE_ROOT),
    "dataset_source": str(COLAB_DATASET_SOURCE),
    "output_root": str(COLAB_OUTPUT_ROOT),
    "stage_data_locally": COLAB_STAGE_DATA_LOCALLY,
    "mixed_precision_policy": str(keras.mixed_precision.global_policy()),
}
(OUTPUT / "environment.json").write_text(
    json.dumps(environment_record, indent=2), encoding="utf-8"
)


## 2. Verify the Drive dataset and stage it to Colab scratch


In [ ]:
def locate_patient_root(search_root):
    """Return a directory containing the 203 PatientID_* folders, if present."""
    search_root = Path(search_root).expanduser().resolve()
    if not search_root.is_dir():
        return None
    direct = list(search_root.glob("PatientID_*"))
    if len(direct) >= 200:
        return search_root
    named = sorted(
        (p for p in search_root.rglob("MU-Glioma-Post") if p.is_dir()),
        key=lambda p: len(p.parts),
    )
    for candidate in named:
        if len(list(candidate.glob("PatientID_*"))) >= 200:
            return candidate
    parent_counts = {}
    for patient_dir in search_root.rglob("PatientID_*"):
        if patient_dir.is_dir():
            parent_counts[patient_dir.parent] = parent_counts.get(patient_dir.parent, 0) + 1
    if parent_counts:
        best_parent, count = max(parent_counts.items(), key=lambda item: item[1])
        if count >= 200:
            return best_parent
    return None


def dataset_inventory(root):
    root = locate_patient_root(root)
    if root is None:
        return None
    return {
        "root": root,
        "patients": len(list(root.glob("PatientID_*"))),
        "tumor_masks": len(list(root.rglob("*_tumorMask.nii.gz"))),
        "brain_t1n": len(list(root.rglob("*_brain_t1n.nii.gz"))),
        "brain_t1c": len(list(root.rglob("*_brain_t1c.nii.gz"))),
        "brain_t2w": len(list(root.rglob("*_brain_t2w.nii.gz"))),
        "brain_t2f": len(list(root.rglob("*_brain_t2f.nii.gz"))),
    }


EXPECTED_SOURCE_INVENTORY = {
    "patients": 203,
    "tumor_masks": 594,
    "brain_t1n": 596,
    "brain_t1c": 596,
    "brain_t2w": 596,
    "brain_t2f": 596,
}


def assert_complete_inventory(inventory, description):
    assert inventory is not None, f"Patient folders not found in {description}"
    actual = {key: inventory[key] for key in EXPECTED_SOURCE_INVENTORY}
    assert actual == EXPECTED_SOURCE_INVENTORY, (
        f"Incomplete or unexpected {description} inventory: {actual}; "
        f"expected {EXPECTED_SOURCE_INVENTORY}"
    )


source_candidates = [
    COLAB_DATASET_SOURCE,
    COLAB_DRIVE_ROOT / "MU-Glioma-Post-Kaggle.bundle",
    COLAB_DRIVE_ROOT / "MU-Glioma-Post.bundle",
    COLAB_DRIVE_ROOT / "MU-Glioma-Post.tar",
    COLAB_DRIVE_ROOT / "MU-Glioma-Post.tar.gz",
]
dataset_source = next((path for path in source_candidates if path.exists()), None)
assert dataset_source is not None, (
    "Dataset source not found. Put the extracted MU-Glioma-Post folder or "
    "MU-Glioma-Post-Kaggle.bundle directly in MyDrive, or change only "
    "COLAB_DATASET_SOURCE in the environment cell."
)

if dataset_source.is_dir():
    source_inventory = dataset_inventory(dataset_source)
    assert_complete_inventory(source_inventory, "Drive dataset")

if COLAB_STAGE_DATA_LOCALLY:
    scratch_dataset_parent = COLAB_SCRATCH_ROOT / "dataset"
    staged_inventory = dataset_inventory(scratch_dataset_parent)
    staged_complete = False
    if staged_inventory is not None:
        staged_actual = {
            key: staged_inventory[key] for key in EXPECTED_SOURCE_INVENTORY
        }
        staged_complete = staged_actual == EXPECTED_SOURCE_INVENTORY

    if not staged_complete:
        if scratch_dataset_parent.exists():
            shutil.rmtree(scratch_dataset_parent)
        scratch_dataset_parent.mkdir(parents=True, exist_ok=True)
        free_bytes = shutil.disk_usage("/content").free
        assert free_bytes >= 20 * 1024**3, (
            f"Only {free_bytes / 1024**3:.1f} GiB free under /content; "
            "at least 20 GiB is required to stage this 12-GB dataset safely."
        )
        if dataset_source.is_dir():
            target = scratch_dataset_parent / "MU-Glioma-Post"
            print("Copying verified dataset from Drive to Colab scratch:", target)
            shutil.copytree(source_inventory["root"], target)
        else:
            print("Extracting", dataset_source, "to Colab scratch ...")
            subprocess.run(
                ["tar", "-xf", str(dataset_source), "-C", str(scratch_dataset_parent)],
                check=True,
            )
        staged_inventory = dataset_inventory(scratch_dataset_parent)
        assert_complete_inventory(staged_inventory, "staged Colab dataset")
    local_data = staged_inventory["root"]
else:
    assert dataset_source.is_dir(), (
        "Direct Drive access requires an extracted dataset directory, not an archive."
    )
    local_data = source_inventory["root"]

assert_complete_inventory(dataset_inventory(local_data), "active dataset")
print("Active dataset root:", local_data)

environment_record = json.loads((OUTPUT / "environment.json").read_text())
environment_record["dataset_source_resolved"] = str(dataset_source)
environment_record["active_dataset_root"] = str(local_data)
environment_record["dataset_inventory"] = {
    key: dataset_inventory(local_data)[key] for key in EXPECTED_SOURCE_INVENTORY
}
(OUTPUT / "environment.json").write_text(
    json.dumps(environment_record, indent=2), encoding="utf-8"
)


## 3. Reconstruct and verify patient fold 3 (SHA-256 locked)


In [ ]:
modalities = ["brain_t1n", "brain_t1c", "brain_t2w", "brain_t2f"]
cases = []
missing_masks = []

for patient_dir in sorted(local_data.glob("PatientID_*")):
    for tp_dir in sorted(patient_dir.glob("Timepoint_*")):
        prefix = f"{patient_dir.name}_{tp_dir.name}"
        mask = tp_dir / f"{prefix}_tumorMask.nii.gz"
        images = [tp_dir / f"{prefix}_{modality}.nii.gz" for modality in modalities]
        if all(path.exists() for path in images) and mask.exists():
            cases.append({
                "id": prefix,
                "patient": patient_dir.name,
                "images": [str(path) for path in images],
                "mask": str(mask),
            })
        elif all(path.exists() for path in images):
            missing_masks.append(prefix)

patients = sorted({case["patient"] for case in cases})
split_record = json.loads(r"""{
  "design": "five_outer_folds_with_patient_grouped_inner_tuning_v2",
  "outer_fold": 3,
  "outer_split_seed": 2026,
  "inner_split_seed": 6205,
  "inner_tuning_fraction_of_outer_training_patients": 0.2,
  "fit_train_patients": [
    "PatientID_0003",
    "PatientID_0005",
    "PatientID_0007",
    "PatientID_0011",
    "PatientID_0012",
    "PatientID_0013",
    "PatientID_0014",
    "PatientID_0015",
    "PatientID_0019",
    "PatientID_0020",
    "PatientID_0022",
    "PatientID_0025",
    "PatientID_0026",
    "PatientID_0029",
    "PatientID_0031",
    "PatientID_0032",
    "PatientID_0034",
    "PatientID_0041",
    "PatientID_0043",
    "PatientID_0044",
    "PatientID_0046",
    "PatientID_0047",
    "PatientID_0049",
    "PatientID_0052",
    "PatientID_0053",
    "PatientID_0055",
    "PatientID_0057",
    "PatientID_0058",
    "PatientID_0059",
    "PatientID_0060",
    "PatientID_0061",
    "PatientID_0062",
    "PatientID_0063",
    "PatientID_0064",
    "PatientID_0065",
    "PatientID_0068",
    "PatientID_0069",
    "PatientID_0071",
    "PatientID_0072",
    "PatientID_0073",
    "PatientID_0074",
    "PatientID_0076",
    "PatientID_0078",
    "PatientID_0079",
    "PatientID_0080",
    "PatientID_0082",
    "PatientID_0083",
    "PatientID_0084",
    "PatientID_0086",
    "PatientID_0087",
    "PatientID_0088",
    "PatientID_0089",
    "PatientID_0090",
    "PatientID_0091",
    "PatientID_0092",
    "PatientID_0093",
    "PatientID_0094",
    "PatientID_0097",
    "PatientID_0101",
    "PatientID_0105",
    "PatientID_0112",
    "PatientID_0113",
    "PatientID_0115",
    "PatientID_0117",
    "PatientID_0118",
    "PatientID_0121",
    "PatientID_0124",
    "PatientID_0125",
    "PatientID_0126",
    "PatientID_0133",
    "PatientID_0135",
    "PatientID_0136",
    "PatientID_0138",
    "PatientID_0139",
    "PatientID_0140",
    "PatientID_0141",
    "PatientID_0143",
    "PatientID_0144",
    "PatientID_0145",
    "PatientID_0149",
    "PatientID_0150",
    "PatientID_0151",
    "PatientID_0155",
    "PatientID_0156",
    "PatientID_0158",
    "PatientID_0159",
    "PatientID_0160",
    "PatientID_0161",
    "PatientID_0162",
    "PatientID_0168",
    "PatientID_0169",
    "PatientID_0171",
    "PatientID_0173",
    "PatientID_0174",
    "PatientID_0186",
    "PatientID_0187",
    "PatientID_0188",
    "PatientID_0192",
    "PatientID_0195",
    "PatientID_0197",
    "PatientID_0202",
    "PatientID_0204",
    "PatientID_0205",
    "PatientID_0206",
    "PatientID_0208",
    "PatientID_0210",
    "PatientID_0211",
    "PatientID_0213",
    "PatientID_0214",
    "PatientID_0215",
    "PatientID_0220",
    "PatientID_0235",
    "PatientID_0236",
    "PatientID_0237",
    "PatientID_0238",
    "PatientID_0239",
    "PatientID_0240",
    "PatientID_0244",
    "PatientID_0249",
    "PatientID_0252",
    "PatientID_0254",
    "PatientID_0256",
    "PatientID_0258",
    "PatientID_0264",
    "PatientID_0265",
    "PatientID_0267",
    "PatientID_0268",
    "PatientID_0272",
    "PatientID_0273",
    "PatientID_0274"
  ],
  "tuning_patients": [
    "PatientID_0004",
    "PatientID_0006",
    "PatientID_0009",
    "PatientID_0023",
    "PatientID_0035",
    "PatientID_0038",
    "PatientID_0039",
    "PatientID_0040",
    "PatientID_0042",
    "PatientID_0070",
    "PatientID_0077",
    "PatientID_0081",
    "PatientID_0095",
    "PatientID_0099",
    "PatientID_0103",
    "PatientID_0109",
    "PatientID_0111",
    "PatientID_0114",
    "PatientID_0123",
    "PatientID_0132",
    "PatientID_0157",
    "PatientID_0189",
    "PatientID_0191",
    "PatientID_0196",
    "PatientID_0200",
    "PatientID_0203",
    "PatientID_0207",
    "PatientID_0250",
    "PatientID_0251",
    "PatientID_0261",
    "PatientID_0271",
    "PatientID_0275"
  ],
  "outer_test_patients": [
    "PatientID_0008",
    "PatientID_0010",
    "PatientID_0018",
    "PatientID_0021",
    "PatientID_0024",
    "PatientID_0027",
    "PatientID_0030",
    "PatientID_0033",
    "PatientID_0036",
    "PatientID_0037",
    "PatientID_0045",
    "PatientID_0051",
    "PatientID_0054",
    "PatientID_0056",
    "PatientID_0066",
    "PatientID_0067",
    "PatientID_0085",
    "PatientID_0106",
    "PatientID_0108",
    "PatientID_0110",
    "PatientID_0116",
    "PatientID_0120",
    "PatientID_0130",
    "PatientID_0142",
    "PatientID_0147",
    "PatientID_0148",
    "PatientID_0152",
    "PatientID_0164",
    "PatientID_0165",
    "PatientID_0170",
    "PatientID_0198",
    "PatientID_0199",
    "PatientID_0201",
    "PatientID_0209",
    "PatientID_0212",
    "PatientID_0216",
    "PatientID_0242",
    "PatientID_0255",
    "PatientID_0259",
    "PatientID_0260",
    "PatientID_0270"
  ],
  "fit_train_cases": [
    "PatientID_0003_Timepoint_1",
    "PatientID_0003_Timepoint_2",
    "PatientID_0003_Timepoint_5",
    "PatientID_0005_Timepoint_3",
    "PatientID_0005_Timepoint_4",
    "PatientID_0007_Timepoint_2",
    "PatientID_0007_Timepoint_3",
    "PatientID_0007_Timepoint_4",
    "PatientID_0007_Timepoint_5",
    "PatientID_0007_Timepoint_6",
    "PatientID_0011_Timepoint_1",
    "PatientID_0011_Timepoint_2",
    "PatientID_0012_Timepoint_2",
    "PatientID_0012_Timepoint_3",
    "PatientID_0013_Timepoint_1",
    "PatientID_0013_Timepoint_2",
    "PatientID_0013_Timepoint_4",
    "PatientID_0013_Timepoint_5",
    "PatientID_0013_Timepoint_6",
    "PatientID_0014_Timepoint_1",
    "PatientID_0014_Timepoint_2",
    "PatientID_0014_Timepoint_3",
    "PatientID_0014_Timepoint_4",
    "PatientID_0014_Timepoint_5",
    "PatientID_0014_Timepoint_6",
    "PatientID_0015_Timepoint_1",
    "PatientID_0019_Timepoint_4",
    "PatientID_0019_Timepoint_5",
    "PatientID_0019_Timepoint_6",
    "PatientID_0020_Timepoint_1",
    "PatientID_0020_Timepoint_2",
    "PatientID_0020_Timepoint_3",
    "PatientID_0020_Timepoint_4",
    "PatientID_0022_Timepoint_1",
    "PatientID_0022_Timepoint_2",
    "PatientID_0022_Timepoint_3",
    "PatientID_0025_Timepoint_1",
    "PatientID_0025_Timepoint_2",
    "PatientID_0025_Timepoint_3",
    "PatientID_0026_Timepoint_1",
    "PatientID_0026_Timepoint_2",
    "PatientID_0029_Timepoint_1",
    "PatientID_0029_Timepoint_3",
    "PatientID_0029_Timepoint_4",
    "PatientID_0031_Timepoint_2",
    "PatientID_0031_Timepoint_3",
    "PatientID_0031_Timepoint_4",
    "PatientID_0031_Timepoint_5",
    "PatientID_0032_Timepoint_1",
    "PatientID_0032_Timepoint_2",
    "PatientID_0032_Timepoint_3",
    "PatientID_0032_Timepoint_4",
    "PatientID_0032_Timepoint_5",
    "PatientID_0032_Timepoint_6",
    "PatientID_0034_Timepoint_1",
    "PatientID_0034_Timepoint_2",
    "PatientID_0034_Timepoint_3",
    "PatientID_0034_Timepoint_4",
    "PatientID_0034_Timepoint_5",
    "PatientID_0034_Timepoint_6",
    "PatientID_0041_Timepoint_1",
    "PatientID_0041_Timepoint_2",
    "PatientID_0043_Timepoint_1",
    "PatientID_0044_Timepoint_1",
    "PatientID_0044_Timepoint_2",
    "PatientID_0044_Timepoint_3",
    "PatientID_0044_Timepoint_4",
    "PatientID_0044_Timepoint_5",
    "PatientID_0044_Timepoint_6",
    "PatientID_0046_Timepoint_1",
    "PatientID_0047_Timepoint_1",
    "PatientID_0049_Timepoint_1",
    "PatientID_0052_Timepoint_1",
    "PatientID_0052_Timepoint_2",
    "PatientID_0053_Timepoint_1",
    "PatientID_0053_Timepoint_3",
    "PatientID_0053_Timepoint_4",
    "PatientID_0053_Timepoint_5",
    "PatientID_0053_Timepoint_6",
    "PatientID_0055_Timepoint_1",
    "PatientID_0055_Timepoint_4",
    "PatientID_0055_Timepoint_5",
    "PatientID_0055_Timepoint_6",
    "PatientID_0057_Timepoint_1",
    "PatientID_0058_Timepoint_1",
    "PatientID_0059_Timepoint_1",
    "PatientID_0059_Timepoint_2",
    "PatientID_0059_Timepoint_3",
    "PatientID_0060_Timepoint_1",
    "PatientID_0060_Timepoint_2",
    "PatientID_0061_Timepoint_2",
    "PatientID_0062_Timepoint_1",
    "PatientID_0062_Timepoint_2",
    "PatientID_0063_Timepoint_1",
    "PatientID_0064_Timepoint_1",
    "PatientID_0064_Timepoint_2",
    "PatientID_0065_Timepoint_1",
    "PatientID_0065_Timepoint_2",
    "PatientID_0065_Timepoint_3",
    "PatientID_0065_Timepoint_4",
    "PatientID_0065_Timepoint_5",
    "PatientID_0068_Timepoint_1",
    "PatientID_0068_Timepoint_2",
    "PatientID_0068_Timepoint_3",
    "PatientID_0068_Timepoint_4",
    "PatientID_0068_Timepoint_5",
    "PatientID_0069_Timepoint_2",
    "PatientID_0069_Timepoint_3",
    "PatientID_0069_Timepoint_4",
    "PatientID_0069_Timepoint_5",
    "PatientID_0069_Timepoint_6",
    "PatientID_0071_Timepoint_1",
    "PatientID_0072_Timepoint_1",
    "PatientID_0072_Timepoint_2",
    "PatientID_0072_Timepoint_3",
    "PatientID_0073_Timepoint_1",
    "PatientID_0073_Timepoint_2",
    "PatientID_0073_Timepoint_3",
    "PatientID_0073_Timepoint_4",
    "PatientID_0074_Timepoint_1",
    "PatientID_0074_Timepoint_2",
    "PatientID_0074_Timepoint_3",
    "PatientID_0074_Timepoint_4",
    "PatientID_0074_Timepoint_5",
    "PatientID_0074_Timepoint_6",
    "PatientID_0076_Timepoint_1",
    "PatientID_0076_Timepoint_2",
    "PatientID_0078_Timepoint_1",
    "PatientID_0078_Timepoint_2",
    "PatientID_0078_Timepoint_3",
    "PatientID_0079_Timepoint_1",
    "PatientID_0079_Timepoint_2",
    "PatientID_0079_Timepoint_3",
    "PatientID_0079_Timepoint_4",
    "PatientID_0079_Timepoint_5",
    "PatientID_0079_Timepoint_6",
    "PatientID_0080_Timepoint_1",
    "PatientID_0080_Timepoint_2",
    "PatientID_0082_Timepoint_1",
    "PatientID_0082_Timepoint_2",
    "PatientID_0083_Timepoint_1",
    "PatientID_0083_Timepoint_2",
    "PatientID_0083_Timepoint_3",
    "PatientID_0083_Timepoint_4",
    "PatientID_0083_Timepoint_5",
    "PatientID_0083_Timepoint_6",
    "PatientID_0084_Timepoint_1",
    "PatientID_0084_Timepoint_2",
    "PatientID_0084_Timepoint_3",
    "PatientID_0084_Timepoint_4",
    "PatientID_0084_Timepoint_5",
    "PatientID_0084_Timepoint_6",
    "PatientID_0086_Timepoint_1",
    "PatientID_0087_Timepoint_1",
    "PatientID_0087_Timepoint_2",
    "PatientID_0087_Timepoint_3",
    "PatientID_0088_Timepoint_1",
    "PatientID_0088_Timepoint_2",
    "PatientID_0089_Timepoint_1",
    "PatientID_0089_Timepoint_2",
    "PatientID_0089_Timepoint_3",
    "PatientID_0089_Timepoint_4",
    "PatientID_0089_Timepoint_5",
    "PatientID_0089_Timepoint_6",
    "PatientID_0090_Timepoint_1",
    "PatientID_0090_Timepoint_2",
    "PatientID_0091_Timepoint_1",
    "PatientID_0091_Timepoint_2",
    "PatientID_0092_Timepoint_1",
    "PatientID_0092_Timepoint_2",
    "PatientID_0093_Timepoint_1",
    "PatientID_0093_Timepoint_2",
    "PatientID_0093_Timepoint_3",
    "PatientID_0093_Timepoint_4",
    "PatientID_0094_Timepoint_1",
    "PatientID_0094_Timepoint_2",
    "PatientID_0094_Timepoint_3",
    "PatientID_0094_Timepoint_4",
    "PatientID_0094_Timepoint_5",
    "PatientID_0094_Timepoint_6",
    "PatientID_0097_Timepoint_1",
    "PatientID_0097_Timepoint_2",
    "PatientID_0097_Timepoint_3",
    "PatientID_0097_Timepoint_4",
    "PatientID_0097_Timepoint_5",
    "PatientID_0097_Timepoint_6",
    "PatientID_0101_Timepoint_1",
    "PatientID_0105_Timepoint_1",
    "PatientID_0105_Timepoint_2",
    "PatientID_0105_Timepoint_3",
    "PatientID_0105_Timepoint_4",
    "PatientID_0105_Timepoint_5",
    "PatientID_0105_Timepoint_6",
    "PatientID_0112_Timepoint_1",
    "PatientID_0113_Timepoint_1",
    "PatientID_0113_Timepoint_2",
    "PatientID_0113_Timepoint_3",
    "PatientID_0115_Timepoint_1",
    "PatientID_0117_Timepoint_1",
    "PatientID_0117_Timepoint_2",
    "PatientID_0117_Timepoint_3",
    "PatientID_0118_Timepoint_1",
    "PatientID_0118_Timepoint_2",
    "PatientID_0118_Timepoint_3",
    "PatientID_0121_Timepoint_1",
    "PatientID_0121_Timepoint_2",
    "PatientID_0121_Timepoint_3",
    "PatientID_0124_Timepoint_1",
    "PatientID_0125_Timepoint_1",
    "PatientID_0125_Timepoint_2",
    "PatientID_0126_Timepoint_1",
    "PatientID_0133_Timepoint_1",
    "PatientID_0135_Timepoint_1",
    "PatientID_0135_Timepoint_2",
    "PatientID_0135_Timepoint_3",
    "PatientID_0136_Timepoint_1",
    "PatientID_0136_Timepoint_2",
    "PatientID_0136_Timepoint_3",
    "PatientID_0138_Timepoint_1",
    "PatientID_0138_Timepoint_2",
    "PatientID_0138_Timepoint_3",
    "PatientID_0139_Timepoint_1",
    "PatientID_0140_Timepoint_1",
    "PatientID_0140_Timepoint_2",
    "PatientID_0140_Timepoint_3",
    "PatientID_0141_Timepoint_1",
    "PatientID_0143_Timepoint_1",
    "PatientID_0144_Timepoint_1",
    "PatientID_0145_Timepoint_1",
    "PatientID_0145_Timepoint_2",
    "PatientID_0149_Timepoint_1",
    "PatientID_0149_Timepoint_2",
    "PatientID_0149_Timepoint_3",
    "PatientID_0149_Timepoint_4",
    "PatientID_0150_Timepoint_1",
    "PatientID_0150_Timepoint_2",
    "PatientID_0150_Timepoint_3",
    "PatientID_0150_Timepoint_4",
    "PatientID_0150_Timepoint_6",
    "PatientID_0151_Timepoint_1",
    "PatientID_0151_Timepoint_2",
    "PatientID_0155_Timepoint_1",
    "PatientID_0156_Timepoint_1",
    "PatientID_0156_Timepoint_2",
    "PatientID_0156_Timepoint_3",
    "PatientID_0158_Timepoint_1",
    "PatientID_0159_Timepoint_1",
    "PatientID_0159_Timepoint_2",
    "PatientID_0159_Timepoint_3",
    "PatientID_0159_Timepoint_4",
    "PatientID_0160_Timepoint_1",
    "PatientID_0160_Timepoint_2",
    "PatientID_0161_Timepoint_1",
    "PatientID_0162_Timepoint_1",
    "PatientID_0162_Timepoint_2",
    "PatientID_0162_Timepoint_3",
    "PatientID_0162_Timepoint_4",
    "PatientID_0162_Timepoint_5",
    "PatientID_0162_Timepoint_6",
    "PatientID_0168_Timepoint_1",
    "PatientID_0169_Timepoint_1",
    "PatientID_0169_Timepoint_2",
    "PatientID_0169_Timepoint_3",
    "PatientID_0169_Timepoint_4",
    "PatientID_0171_Timepoint_1",
    "PatientID_0171_Timepoint_2",
    "PatientID_0171_Timepoint_3",
    "PatientID_0173_Timepoint_1",
    "PatientID_0173_Timepoint_2",
    "PatientID_0174_Timepoint_1",
    "PatientID_0186_Timepoint_1",
    "PatientID_0186_Timepoint_2",
    "PatientID_0186_Timepoint_3",
    "PatientID_0186_Timepoint_4",
    "PatientID_0186_Timepoint_5",
    "PatientID_0186_Timepoint_6",
    "PatientID_0187_Timepoint_1",
    "PatientID_0187_Timepoint_2",
    "PatientID_0188_Timepoint_1",
    "PatientID_0188_Timepoint_2",
    "PatientID_0192_Timepoint_1",
    "PatientID_0192_Timepoint_2",
    "PatientID_0192_Timepoint_3",
    "PatientID_0195_Timepoint_1",
    "PatientID_0195_Timepoint_2",
    "PatientID_0195_Timepoint_3",
    "PatientID_0197_Timepoint_1",
    "PatientID_0197_Timepoint_2",
    "PatientID_0197_Timepoint_3",
    "PatientID_0202_Timepoint_1",
    "PatientID_0202_Timepoint_2",
    "PatientID_0202_Timepoint_3",
    "PatientID_0204_Timepoint_1",
    "PatientID_0204_Timepoint_2",
    "PatientID_0204_Timepoint_3",
    "PatientID_0204_Timepoint_4",
    "PatientID_0204_Timepoint_5",
    "PatientID_0204_Timepoint_6",
    "PatientID_0205_Timepoint_1",
    "PatientID_0205_Timepoint_2",
    "PatientID_0205_Timepoint_3",
    "PatientID_0205_Timepoint_4",
    "PatientID_0205_Timepoint_5",
    "PatientID_0206_Timepoint_1",
    "PatientID_0208_Timepoint_1",
    "PatientID_0208_Timepoint_2",
    "PatientID_0210_Timepoint_1",
    "PatientID_0210_Timepoint_2",
    "PatientID_0210_Timepoint_3",
    "PatientID_0210_Timepoint_4",
    "PatientID_0210_Timepoint_6",
    "PatientID_0211_Timepoint_1",
    "PatientID_0211_Timepoint_2",
    "PatientID_0211_Timepoint_3",
    "PatientID_0213_Timepoint_1",
    "PatientID_0213_Timepoint_2",
    "PatientID_0213_Timepoint_3",
    "PatientID_0213_Timepoint_4",
    "PatientID_0213_Timepoint_5",
    "PatientID_0214_Timepoint_1",
    "PatientID_0214_Timepoint_2",
    "PatientID_0214_Timepoint_3",
    "PatientID_0214_Timepoint_4",
    "PatientID_0214_Timepoint_5",
    "PatientID_0214_Timepoint_6",
    "PatientID_0215_Timepoint_1",
    "PatientID_0215_Timepoint_2",
    "PatientID_0215_Timepoint_3",
    "PatientID_0215_Timepoint_4",
    "PatientID_0215_Timepoint_5",
    "PatientID_0215_Timepoint_6",
    "PatientID_0220_Timepoint_1",
    "PatientID_0220_Timepoint_6",
    "PatientID_0235_Timepoint_1",
    "PatientID_0235_Timepoint_3",
    "PatientID_0235_Timepoint_6",
    "PatientID_0236_Timepoint_1",
    "PatientID_0236_Timepoint_3",
    "PatientID_0236_Timepoint_6",
    "PatientID_0237_Timepoint_1",
    "PatientID_0237_Timepoint_3",
    "PatientID_0238_Timepoint_1",
    "PatientID_0238_Timepoint_3",
    "PatientID_0238_Timepoint_6",
    "PatientID_0239_Timepoint_1",
    "PatientID_0239_Timepoint_3",
    "PatientID_0239_Timepoint_6",
    "PatientID_0240_Timepoint_1",
    "PatientID_0240_Timepoint_3",
    "PatientID_0240_Timepoint_6",
    "PatientID_0244_Timepoint_1",
    "PatientID_0249_Timepoint_1",
    "PatientID_0249_Timepoint_3",
    "PatientID_0252_Timepoint_1",
    "PatientID_0252_Timepoint_2",
    "PatientID_0252_Timepoint_3",
    "PatientID_0254_Timepoint_1",
    "PatientID_0254_Timepoint_2",
    "PatientID_0254_Timepoint_3",
    "PatientID_0254_Timepoint_4",
    "PatientID_0256_Timepoint_1",
    "PatientID_0256_Timepoint_2",
    "PatientID_0256_Timepoint_3",
    "PatientID_0256_Timepoint_4",
    "PatientID_0256_Timepoint_5",
    "PatientID_0258_Timepoint_1",
    "PatientID_0264_Timepoint_1",
    "PatientID_0264_Timepoint_2",
    "PatientID_0265_Timepoint_1",
    "PatientID_0265_Timepoint_2",
    "PatientID_0267_Timepoint_1",
    "PatientID_0267_Timepoint_2",
    "PatientID_0268_Timepoint_1",
    "PatientID_0268_Timepoint_3",
    "PatientID_0268_Timepoint_6",
    "PatientID_0272_Timepoint_1",
    "PatientID_0272_Timepoint_3",
    "PatientID_0272_Timepoint_6",
    "PatientID_0273_Timepoint_1",
    "PatientID_0274_Timepoint_1",
    "PatientID_0274_Timepoint_3"
  ],
  "tuning_cases": [
    "PatientID_0004_Timepoint_1",
    "PatientID_0006_Timepoint_2",
    "PatientID_0006_Timepoint_4",
    "PatientID_0006_Timepoint_5",
    "PatientID_0006_Timepoint_6",
    "PatientID_0009_Timepoint_2",
    "PatientID_0023_Timepoint_2",
    "PatientID_0035_Timepoint_1",
    "PatientID_0035_Timepoint_2",
    "PatientID_0035_Timepoint_3",
    "PatientID_0038_Timepoint_1",
    "PatientID_0038_Timepoint_2",
    "PatientID_0038_Timepoint_3",
    "PatientID_0038_Timepoint_4",
    "PatientID_0038_Timepoint_5",
    "PatientID_0038_Timepoint_6",
    "PatientID_0039_Timepoint_1",
    "PatientID_0039_Timepoint_2",
    "PatientID_0039_Timepoint_3",
    "PatientID_0039_Timepoint_4",
    "PatientID_0039_Timepoint_6",
    "PatientID_0040_Timepoint_1",
    "PatientID_0042_Timepoint_1",
    "PatientID_0070_Timepoint_1",
    "PatientID_0070_Timepoint_2",
    "PatientID_0070_Timepoint_3",
    "PatientID_0070_Timepoint_4",
    "PatientID_0077_Timepoint_1",
    "PatientID_0081_Timepoint_1",
    "PatientID_0081_Timepoint_2",
    "PatientID_0095_Timepoint_1",
    "PatientID_0095_Timepoint_2",
    "PatientID_0095_Timepoint_3",
    "PatientID_0095_Timepoint_4",
    "PatientID_0095_Timepoint_6",
    "PatientID_0099_Timepoint_1",
    "PatientID_0099_Timepoint_2",
    "PatientID_0103_Timepoint_1",
    "PatientID_0103_Timepoint_2",
    "PatientID_0103_Timepoint_3",
    "PatientID_0103_Timepoint_4",
    "PatientID_0103_Timepoint_5",
    "PatientID_0103_Timepoint_6",
    "PatientID_0109_Timepoint_1",
    "PatientID_0109_Timepoint_2",
    "PatientID_0109_Timepoint_3",
    "PatientID_0109_Timepoint_4",
    "PatientID_0111_Timepoint_1",
    "PatientID_0114_Timepoint_1",
    "PatientID_0114_Timepoint_2",
    "PatientID_0123_Timepoint_1",
    "PatientID_0123_Timepoint_2",
    "PatientID_0132_Timepoint_1",
    "PatientID_0132_Timepoint_2",
    "PatientID_0132_Timepoint_3",
    "PatientID_0157_Timepoint_1",
    "PatientID_0157_Timepoint_2",
    "PatientID_0157_Timepoint_3",
    "PatientID_0157_Timepoint_4",
    "PatientID_0157_Timepoint_5",
    "PatientID_0157_Timepoint_6",
    "PatientID_0189_Timepoint_1",
    "PatientID_0189_Timepoint_2",
    "PatientID_0189_Timepoint_3",
    "PatientID_0189_Timepoint_4",
    "PatientID_0189_Timepoint_5",
    "PatientID_0189_Timepoint_6",
    "PatientID_0191_Timepoint_2",
    "PatientID_0191_Timepoint_3",
    "PatientID_0191_Timepoint_4",
    "PatientID_0196_Timepoint_1",
    "PatientID_0196_Timepoint_2",
    "PatientID_0196_Timepoint_3",
    "PatientID_0200_Timepoint_1",
    "PatientID_0203_Timepoint_1",
    "PatientID_0207_Timepoint_1",
    "PatientID_0207_Timepoint_2",
    "PatientID_0207_Timepoint_3",
    "PatientID_0207_Timepoint_4",
    "PatientID_0207_Timepoint_5",
    "PatientID_0207_Timepoint_6",
    "PatientID_0250_Timepoint_1",
    "PatientID_0250_Timepoint_2",
    "PatientID_0250_Timepoint_3",
    "PatientID_0251_Timepoint_1",
    "PatientID_0251_Timepoint_2",
    "PatientID_0251_Timepoint_3",
    "PatientID_0261_Timepoint_1",
    "PatientID_0271_Timepoint_1",
    "PatientID_0271_Timepoint_3",
    "PatientID_0271_Timepoint_6",
    "PatientID_0275_Timepoint_1",
    "PatientID_0275_Timepoint_3",
    "PatientID_0275_Timepoint_6"
  ],
  "outer_test_cases": [
    "PatientID_0008_Timepoint_4",
    "PatientID_0008_Timepoint_6",
    "PatientID_0010_Timepoint_1",
    "PatientID_0010_Timepoint_4",
    "PatientID_0010_Timepoint_5",
    "PatientID_0018_Timepoint_1",
    "PatientID_0018_Timepoint_2",
    "PatientID_0018_Timepoint_4",
    "PatientID_0018_Timepoint_5",
    "PatientID_0018_Timepoint_6",
    "PatientID_0021_Timepoint_2",
    "PatientID_0021_Timepoint_3",
    "PatientID_0021_Timepoint_4",
    "PatientID_0021_Timepoint_5",
    "PatientID_0021_Timepoint_6",
    "PatientID_0024_Timepoint_2",
    "PatientID_0024_Timepoint_3",
    "PatientID_0027_Timepoint_1",
    "PatientID_0030_Timepoint_1",
    "PatientID_0030_Timepoint_3",
    "PatientID_0030_Timepoint_4",
    "PatientID_0033_Timepoint_1",
    "PatientID_0033_Timepoint_2",
    "PatientID_0033_Timepoint_3",
    "PatientID_0033_Timepoint_4",
    "PatientID_0033_Timepoint_5",
    "PatientID_0036_Timepoint_1",
    "PatientID_0036_Timepoint_2",
    "PatientID_0036_Timepoint_3",
    "PatientID_0036_Timepoint_4",
    "PatientID_0036_Timepoint_5",
    "PatientID_0036_Timepoint_6",
    "PatientID_0037_Timepoint_1",
    "PatientID_0037_Timepoint_2",
    "PatientID_0037_Timepoint_3",
    "PatientID_0037_Timepoint_4",
    "PatientID_0045_Timepoint_1",
    "PatientID_0045_Timepoint_2",
    "PatientID_0045_Timepoint_3",
    "PatientID_0045_Timepoint_4",
    "PatientID_0051_Timepoint_1",
    "PatientID_0051_Timepoint_3",
    "PatientID_0051_Timepoint_4",
    "PatientID_0054_Timepoint_1",
    "PatientID_0054_Timepoint_2",
    "PatientID_0056_Timepoint_2",
    "PatientID_0066_Timepoint_1",
    "PatientID_0066_Timepoint_2",
    "PatientID_0066_Timepoint_3",
    "PatientID_0066_Timepoint_5",
    "PatientID_0066_Timepoint_6",
    "PatientID_0067_Timepoint_1",
    "PatientID_0067_Timepoint_2",
    "PatientID_0085_Timepoint_1",
    "PatientID_0085_Timepoint_2",
    "PatientID_0085_Timepoint_3",
    "PatientID_0085_Timepoint_4",
    "PatientID_0106_Timepoint_2",
    "PatientID_0108_Timepoint_1",
    "PatientID_0110_Timepoint_1",
    "PatientID_0110_Timepoint_2",
    "PatientID_0110_Timepoint_3",
    "PatientID_0116_Timepoint_1",
    "PatientID_0116_Timepoint_2",
    "PatientID_0120_Timepoint_1",
    "PatientID_0130_Timepoint_1",
    "PatientID_0142_Timepoint_1",
    "PatientID_0142_Timepoint_2",
    "PatientID_0147_Timepoint_1",
    "PatientID_0147_Timepoint_2",
    "PatientID_0147_Timepoint_3",
    "PatientID_0148_Timepoint_1",
    "PatientID_0148_Timepoint_2",
    "PatientID_0152_Timepoint_1",
    "PatientID_0152_Timepoint_2",
    "PatientID_0152_Timepoint_3",
    "PatientID_0164_Timepoint_1",
    "PatientID_0164_Timepoint_2",
    "PatientID_0164_Timepoint_3",
    "PatientID_0164_Timepoint_4",
    "PatientID_0164_Timepoint_5",
    "PatientID_0165_Timepoint_1",
    "PatientID_0170_Timepoint_1",
    "PatientID_0170_Timepoint_2",
    "PatientID_0198_Timepoint_1",
    "PatientID_0198_Timepoint_2",
    "PatientID_0198_Timepoint_3",
    "PatientID_0198_Timepoint_4",
    "PatientID_0199_Timepoint_1",
    "PatientID_0199_Timepoint_2",
    "PatientID_0199_Timepoint_3",
    "PatientID_0201_Timepoint_1",
    "PatientID_0201_Timepoint_2",
    "PatientID_0201_Timepoint_3",
    "PatientID_0201_Timepoint_4",
    "PatientID_0201_Timepoint_5",
    "PatientID_0201_Timepoint_6",
    "PatientID_0209_Timepoint_1",
    "PatientID_0209_Timepoint_2",
    "PatientID_0209_Timepoint_3",
    "PatientID_0209_Timepoint_4",
    "PatientID_0209_Timepoint_5",
    "PatientID_0209_Timepoint_6",
    "PatientID_0212_Timepoint_1",
    "PatientID_0212_Timepoint_2",
    "PatientID_0212_Timepoint_3",
    "PatientID_0216_Timepoint_1",
    "PatientID_0242_Timepoint_1",
    "PatientID_0242_Timepoint_3",
    "PatientID_0242_Timepoint_6",
    "PatientID_0255_Timepoint_1",
    "PatientID_0255_Timepoint_2",
    "PatientID_0259_Timepoint_1",
    "PatientID_0259_Timepoint_2",
    "PatientID_0260_Timepoint_1",
    "PatientID_0260_Timepoint_2",
    "PatientID_0270_Timepoint_1",
    "PatientID_0270_Timepoint_3",
    "PatientID_0270_Timepoint_6"
  ],
  "role_balance_diagnostics": {
    "fit_train": {
      "patients": 130,
      "scans": 381,
      "label_present_scans": {
        "1": 202,
        "2": 378,
        "3": 350,
        "4": 334
      },
      "label_absent_scans": {
        "1": 179,
        "2": 3,
        "3": 31,
        "4": 47
      }
    },
    "tuning": {
      "patients": 32,
      "scans": 94,
      "label_present_scans": {
        "1": 50,
        "2": 94,
        "3": 86,
        "4": 82
      },
      "label_absent_scans": {
        "1": 44,
        "2": 0,
        "3": 8,
        "4": 12
      }
    },
    "outer_test": {
      "patients": 41,
      "scans": 119,
      "label_present_scans": {
        "1": 63,
        "2": 119,
        "3": 110,
        "4": 104
      },
      "label_absent_scans": {
        "1": 56,
        "2": 0,
        "3": 9,
        "4": 15
      }
    }
  },
  "missing_masks_excluded": [
    "PatientID_0187_Timepoint_3",
    "PatientID_0191_Timepoint_1"
  ]
}""")
fit_train_patient_set = set(split_record["fit_train_patients"])
tuning_patient_set = set(split_record["tuning_patients"])
outer_test_patient_set = set(split_record["outer_test_patients"])
train_cases = [case for case in cases if case["patient"] in fit_train_patient_set]
tuning_cases = [case for case in cases if case["patient"] in tuning_patient_set]
outer_test_cases = [case for case in cases if case["patient"] in outer_test_patient_set]
# The inherited Keras training cell uses val_cases only for model optimization.
val_cases = tuning_cases

split_text = json.dumps(split_record, indent=2)
split_sha256 = hashlib.sha256(split_text.encode("utf-8")).hexdigest()

assert len(cases) == 594, f"Expected 594 labelled cases; found {len(cases)}"
assert len(patients) == 203, f"Expected 203 patients; found {len(patients)}"
assert missing_masks == split_record["missing_masks_excluded"]
assert len(train_cases) == EXPECTED_FIT_TRAIN_CASES
assert len(fit_train_patient_set) == EXPECTED_FIT_TRAIN_PATIENTS
assert len(tuning_cases) == EXPECTED_TUNING_CASES
assert len(tuning_patient_set) == EXPECTED_TUNING_PATIENTS
assert len(outer_test_cases) == EXPECTED_OUTER_TEST_CASES
assert len(outer_test_patient_set) == EXPECTED_OUTER_TEST_PATIENTS
assert [case["id"] for case in train_cases] == split_record["fit_train_cases"]
assert [case["id"] for case in tuning_cases] == split_record["tuning_cases"]
assert [case["id"] for case in outer_test_cases] == split_record["outer_test_cases"]
assert fit_train_patient_set.isdisjoint(tuning_patient_set)
assert fit_train_patient_set.isdisjoint(outer_test_patient_set)
assert tuning_patient_set.isdisjoint(outer_test_patient_set)
assert fit_train_patient_set | tuning_patient_set | outer_test_patient_set == set(patients)
assert split_sha256 == EXPECTED_SPLIT_SHA256, (
    f"Fold checksum mismatch: {split_sha256}. Stop: this is not the prespecified split."
)

(OUTPUT / "patient_split.json").write_text(split_text, encoding="utf-8")
print("Run:", RUN_ID, "/ fold:", CV_FOLD, "/ training seed:", TRAINING_SEED)
print(f"All labelled cases: {len(cases)} / patients: {len(patients)}")
print(f"Fit training: {len(train_cases)} scans / {len(fit_train_patient_set)} patients")
print(f"Tuning only: {len(tuning_cases)} scans / {len(tuning_patient_set)} patients")
print(f"Outer test only: {len(outer_test_cases)} scans / {len(outer_test_patient_set)} patients")
print("Split SHA-256:", split_sha256)


## 4. Full dataset geometry and label preflight


In [ ]:
# Check every labelled timepoint before starting an expensive training run.
geometry_errors = []
observed_labels = set()
class_presence = {label: 0 for label in range(NUM_CLASSES)}
started = time.time()

for number, case in enumerate(cases, 1):
    image_headers = [nib.load(path) for path in case["images"]]
    mask_image = nib.load(case["mask"])
    reference = image_headers[0]
    for candidate in image_headers[1:] + [mask_image]:
        if candidate.shape != reference.shape or not np.allclose(
            candidate.affine, reference.affine, atol=1e-3
        ):
            geometry_errors.append(case["id"])
            break
    labels_in_case = set(map(int, np.unique(np.asanyarray(mask_image.dataobj))))
    observed_labels.update(labels_in_case)
    for label in labels_in_case:
        class_presence[label] = class_presence.get(label, 0) + 1
    if number % 100 == 0:
        print(f"Checked {number}/{len(cases)} scans")

assert not geometry_errors, f"Geometry mismatch in: {geometry_errors[:10]}"
assert observed_labels.issubset(set(range(NUM_CLASSES))), (
    f"Unexpected labels: {sorted(observed_labels)}"
)
print(f"Preflight passed in {(time.time() - started) / 60:.1f} min")
print("Observed mask labels:", sorted(observed_labels))
print("Scans containing each label:", class_presence)


## 5. Shared normalization, sampling, and augmentation pipeline


In [ ]:
def normalize_mri(array):
    array = array.astype(np.float32, copy=False)
    foreground = array != 0
    if foreground.any():
        values = array[foreground]
        mean = float(values.mean())
        std = float(values.std())
        array = array.copy()
        array[foreground] = (values - mean) / max(std, 1e-6)
        array[~foreground] = 0
    return array


def load_case(case):
    channels = [
        normalize_mri(np.asanyarray(nib.load(path).dataobj))
        for path in case["images"]
    ]
    image = np.stack(channels, axis=-1).astype(np.float32)
    mask = np.asanyarray(nib.load(case["mask"]).dataobj).astype(np.uint8)[..., None]
    return image, mask


def crop_patch(image, mask, center):
    starts = []
    for coordinate, size, patch in zip(center, image.shape[:3], PATCH_SIZE):
        starts.append(int(np.clip(coordinate - patch // 2, 0, max(0, size - patch))))
    slices = tuple(slice(start, start + patch) for start, patch in zip(starts, PATCH_SIZE))
    x_patch, y_patch = image[slices], mask[slices]
    pads = [(0, max(0, patch - size)) for size, patch in zip(x_patch.shape[:3], PATCH_SIZE)]
    if any(after for _, after in pads):
        x_patch = np.pad(x_patch, pads + [(0, 0)])
        y_patch = np.pad(y_patch, pads + [(0, 0)])
    return x_patch.astype(np.float32), y_patch.astype(np.uint8)


def patch_stream(case_list, augment, seed):
    rng = np.random.default_rng(seed)
    while True:
        for case_index in rng.permutation(len(case_list)):
            image, mask = load_case(case_list[case_index])
            class_voxels = {
                label: np.flatnonzero(mask[..., 0] == label)
                for label in range(1, NUM_CLASSES)
            }
            present_classes = [
                label for label, indices in class_voxels.items() if len(indices)
            ]
            for _ in range(PATCHES_PER_CASE):
                if present_classes and rng.random() < 0.80:
                    sampled_class = int(rng.choice(present_classes))
                    flat_index = int(rng.choice(class_voxels[sampled_class]))
                    center = list(np.unravel_index(flat_index, mask.shape[:3]))
                    center = [
                        coordinate + int(rng.integers(-16, 17))
                        for coordinate in center
                    ]
                else:
                    center = [int(rng.integers(0, size)) for size in image.shape[:3]]

                x_patch, y_patch = crop_patch(image, mask, center)
                if augment:
                    for axis in range(3):
                        if rng.random() < 0.5:
                            x_patch = np.flip(x_patch, axis=axis)
                            y_patch = np.flip(y_patch, axis=axis)
                    if rng.random() < 0.30:
                        brain = np.any(x_patch != 0, axis=-1, keepdims=True)
                        changed = (
                            x_patch * rng.uniform(0.9, 1.1)
                            + rng.normal(0, 0.03, x_patch.shape).astype(np.float32)
                        )
                        x_patch = np.where(brain, changed, 0.0)

                y_one_hot = np.eye(NUM_CLASSES, dtype=np.float32)[
                    y_patch[..., 0].astype(np.uint8)
                ]
                yield np.ascontiguousarray(x_patch), np.ascontiguousarray(y_one_hot)


output_signature = (
    tf.TensorSpec(PATCH_SIZE + (4,), tf.float32),
    tf.TensorSpec(PATCH_SIZE + (NUM_CLASSES,), tf.float32),
)
train_ds = tf.data.Dataset.from_generator(
    lambda: patch_stream(train_cases, True, SEED),
    output_signature=output_signature,
).batch(BATCH_SIZE).prefetch(1)
val_ds = tf.data.Dataset.from_generator(
    lambda: patch_stream(val_cases, False, SEED + 1),
    output_signature=output_signature,
).batch(BATCH_SIZE).prefetch(1)

sample_x, sample_y = next(iter(train_ds))
print("Patch tensors:", sample_x.shape, sample_y.shape)
print("Foreground fraction:", float(tf.reduce_mean(1.0 - sample_y[..., 0])))


## 6. Build Adaptive Standard 3D U-Net

            This is the verified plain four-level 3D U-Net with filter widths
16/32/64/128/256, ordinary two-convolution blocks, long skips, and
the shared five-class output.


In [ ]:
def normalization(x, filters):
    if hasattr(layers, "GroupNormalization"):
        return layers.GroupNormalization(groups=min(8, filters), axis=-1)(x)
    return layers.LayerNormalization(axis=-1)(x)


def conv_block(x, filters, dropout=0.0):
    for _ in range(2):
        x = layers.Conv3D(
            filters,
            3,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
        )(x)
        x = normalization(x, filters)
        x = layers.Activation("relu")(x)
    if dropout:
        x = layers.SpatialDropout3D(dropout)(x)
    return x


def build_model(input_shape=PATCH_SIZE + (4,), base_filters=16):
    inputs = keras.Input(input_shape, name="four_modality_mri")
    skips = []
    x = inputs

    for level, filters in enumerate(
        [base_filters, base_filters * 2, base_filters * 4, base_filters * 8]
    ):
        x = conv_block(x, filters, dropout=0.0 if level < 2 else 0.1)
        skips.append(x)
        x = layers.MaxPool3D(2)(x)

    x = conv_block(x, base_filters * 16, dropout=0.2)

    for filters, skip in zip(
        [base_filters * 8, base_filters * 4, base_filters * 2, base_filters],
        reversed(skips),
    ):
        x = layers.Conv3DTranspose(filters, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, skip])
        x = conv_block(
            x,
            filters,
            dropout=0.1 if filters >= base_filters * 4 else 0.0,
        )

    outputs = layers.Conv3D(
        NUM_CLASSES,
        1,
        activation="softmax",
        dtype="float32",
        name="tissue_probabilities",
    )(x)
    return keras.Model(inputs, outputs, name="Adaptive_Standard_3D_UNet")


@keras.utils.register_keras_serializable()
def mean_foreground_dice(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = tuple(range(len(y_pred.shape) - 1))
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denominator = tf.reduce_sum(y_true + y_pred, axis=axes)
    class_dice = (2.0 * intersection + 1e-5) / (denominator + 1e-5)
    present = tf.cast(tf.reduce_sum(y_true, axis=axes) > 0, tf.float32)
    foreground_present = present[1:]
    return tf.math.divide_no_nan(
        tf.reduce_sum(class_dice[1:] * foreground_present),
        tf.reduce_sum(foreground_present),
    )


@keras.utils.register_keras_serializable()
def categorical_ce_dice_loss(y_true, y_pred):
    cross_entropy = tf.reduce_mean(
        keras.losses.categorical_crossentropy(y_true, y_pred)
    )
    return cross_entropy + (1.0 - mean_foreground_dice(y_true, y_pred))


def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss=categorical_ce_dice_loss,
        metrics=[
            mean_foreground_dice,
            keras.metrics.CategoricalAccuracy(name="voxel_accuracy"),
        ],
    )
    return model


model = compile_model(build_model())
model.summary()

# One real gradient update catches graph, shape, numerical, and GPU-memory errors now.
try:
    smoke_metrics = model.train_on_batch(sample_x, sample_y, return_dict=True)
except tf.errors.ResourceExhaustedError as error:
    raise RuntimeError(
        "The locked 96^3, batch-1 configuration does not fit this GPU. "
        "Use a Kaggle P100 or equivalent; do not silently reduce the controlled settings."
    ) from error
assert all(np.isfinite(float(value)) for value in smoke_metrics.values()), smoke_metrics
print("GPU forward/backward smoke test passed:", smoke_metrics)
del model
keras.backend.clear_session()
model = compile_model(build_model())


## 7. Freeze the run configuration


In [ ]:
run_config = {
    "model": MODEL_DISPLAY_NAME,
    "architecture_id": ARCHITECTURE_ID,
    "architecture_description": 'the verified plain four-level Keras 3D U-Net',
    "run_id": RUN_ID,
    "cv_fold": CV_FOLD,
    "split_seed": SPLIT_SEED,
    "inner_split_seed": INNER_SPLIT_SEED,
    "training_seed": TRAINING_SEED,
    "archive_contract_version": "mu_glioma_detailed_zip_v2_nested",
    "checkpoint_selection_role": "patient_grouped_inner_tuning",
    "final_evaluation_role": "untouched_outer_internal_test",
    "fit_train_cases": EXPECTED_FIT_TRAIN_CASES,
    "tuning_cases": EXPECTED_TUNING_CASES,
    "outer_test_cases": EXPECTED_OUTER_TEST_CASES,
    "surface_dice_tolerance_mm": 1.0,
    "lesion_connectivity": "26-connected",
    "split_sha256": split_sha256,
    "modalities_in_channel_order": modalities,
    "labels": {
        "0": "background",
        "1": "non-enhancing tumor core",
        "2": "FLAIR hyperintensity / edema",
        "3": "enhancing tissue",
        "4": "resection cavity",
    },
    "patch_size": list(PATCH_SIZE),
    "batch_size": BATCH_SIZE,
    "patches_per_case": PATCHES_PER_CASE,
    "foreground_centered_patch_probability": 0.80,
    "center_jitter_voxels": 16,
    "maximum_epochs": EPOCHS,
    "early_stopping_monitor": "val_mean_foreground_dice",
    "early_stopping_mode": "max",
    "early_stopping_min_delta": MIN_IMPROVEMENT,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_start_epoch": EARLY_STOPPING_START_EPOCH,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "validation_steps": VALIDATION_STEPS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "categorical cross-entropy + 1 - mean present-class soft foreground Dice",
    "inference_patch_size": list(PATCH_SIZE),
    "inference_stride": list(INFERENCE_STRIDE),
    "save_all_validation_predictions": SAVE_PREDICTIONS,
    "deep_supervision": False,
}
(OUTPUT / "model_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)
print(json.dumps(run_config, indent=2))


## 8. Train with persistent Drive checkpointing and interruption recovery

`BackupAndRestore` writes completed epochs into the unique Google Drive output
folder. After a Colab interruption, reconnect the same Drive, stage the data
again, and use **Run all**; training resumes from the saved epoch. The best
inner-tuning foreground-Dice checkpoint is restored before final serialization.
Do not change locked constants to fit an assigned GPU.


In [ ]:

BEST_WEIGHTS = OUTPUT / "best.weights.h5"
BEST_MODEL = OUTPUT / MODEL_FILE
TRAINING_LOG = OUTPUT / "training_log.csv"
EFFICIENCY_PATH = OUTPUT / "efficiency.json"
EPOCH_TELEMETRY_PATH = OUTPUT / "epoch_telemetry.csv"
FIT_COMPLETE_PATH = OUTPUT / "fit_complete.json"


def current_gpu_memory():
    try:
        return {
            key + "_bytes": int(value)
            for key, value in tf.config.experimental.get_memory_info("GPU:0").items()
        }
    except Exception as error:
        return {"gpu_memory_query_error": repr(error)}


class ResourceTelemetry(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        prior = (
            json.loads(EFFICIENCY_PATH.read_text())
            if EFFICIENCY_PATH.exists()
            else {}
        )
        self.prior_seconds = float(prior.get("training_wall_seconds", 0.0))
        self.session_started = time.perf_counter()
        self.epoch_started = None
        self.rows = (
            pd.read_csv(EPOCH_TELEMETRY_PATH).to_dict(orient="records")
            if EPOCH_TELEMETRY_PATH.exists()
            else []
        )
        try:
            tf.config.experimental.reset_memory_stats("GPU:0")
        except Exception:
            pass

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_started = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        epoch_seconds = time.perf_counter() - self.epoch_started
        memory = current_gpu_memory()
        self.rows.append(
            {
                "epoch_zero_based": int(epoch),
                "epoch_wall_seconds": float(epoch_seconds),
                "gpu_current_bytes": memory.get("current_bytes"),
                "gpu_peak_bytes": memory.get("peak_bytes"),
            }
        )
        pd.DataFrame(self.rows).drop_duplicates(
            subset=["epoch_zero_based"], keep="last"
        ).sort_values("epoch_zero_based").to_csv(EPOCH_TELEMETRY_PATH, index=False)
        elapsed = self.prior_seconds + time.perf_counter() - self.session_started
        efficiency = {
            "run_id": RUN_ID,
            "model": MODEL_DISPLAY_NAME,
            "training_wall_seconds": float(elapsed),
            "total_parameters": int(model.count_params()),
            "trainable_parameters": int(
                sum(np.prod(variable.shape) for variable in model.trainable_weights)
            ),
            "non_trainable_parameters": int(
                sum(np.prod(variable.shape) for variable in model.non_trainable_weights)
            ),
            "peak_gpu_memory_bytes": memory.get("peak_bytes"),
            "gpu_memory_query_error": memory.get("gpu_memory_query_error"),
            "hardware": [str(device) for device in gpu_devices],
            "timing_scope": "model.fit wall time, accumulated across completed epochs/resumes",
        }
        EFFICIENCY_PATH.write_text(json.dumps(efficiency, indent=2))


class FitCompletionMarker(keras.callbacks.Callback):
    """Written only after model.fit completes all callback shutdown cleanly."""

    def on_train_end(self, logs=None):
        FIT_COMPLETE_PATH.write_text(
            json.dumps({"status": "fit_complete", "run_id": RUN_ID}, indent=2),
            encoding="utf-8",
        )


telemetry = ResourceTelemetry()
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_WEIGHTS),
        monitor="val_mean_foreground_dice",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mean_foreground_dice",
        mode="max",
        min_delta=MIN_IMPROVEMENT,
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        start_from_epoch=EARLY_STOPPING_START_EPOCH,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(TRAINING_LOG), append=True),
    telemetry,
    keras.callbacks.BackupAndRestore(
        str(OUTPUT / "training_backup"), save_freq="epoch"
    ),
    FitCompletionMarker(),
]

if FIT_COMPLETE_PATH.exists():
    print("Completed fit marker found; reconstructing the frozen best model.")
else:
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=STEPS_PER_EPOCH,
        validation_steps=VALIDATION_STEPS,
        callbacks=callbacks,
        verbose=1,
    )
    history_serializable = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }
    (OUTPUT / "history.json").write_text(
        json.dumps(history_serializable, indent=2), encoding="utf-8"
    )

assert FIT_COMPLETE_PATH.exists(), "model.fit did not finish cleanly."
assert BEST_WEIGHTS.exists(), "Training ended without a best tuning checkpoint."
model.load_weights(str(BEST_WEIGHTS))
# Save atomically so an interrupted serialization can never be mistaken for a
# completed best model on the next Run All.
temporary_best_model = OUTPUT / f"{BEST_MODEL.stem}.temporary.keras"
if temporary_best_model.exists():
    temporary_best_model.unlink()
model.save(str(temporary_best_model))
os.replace(temporary_best_model, BEST_MODEL)
print("Best inner-tuning-selected model saved to:", BEST_MODEL)
assert BEST_MODEL.exists()
assert TRAINING_LOG.exists()
assert EFFICIENCY_PATH.exists()
efficiency = json.loads(EFFICIENCY_PATH.read_text())
efficiency["best_model_bytes"] = BEST_MODEL.stat().st_size
EFFICIENCY_PATH.write_text(json.dumps(efficiency, indent=2))


## 9. One-time full-volume inference on all 119 untouched outer-test scans


In [ ]:
"""Uniform detailed evaluation helpers for all MU-Glioma segmentation runs."""

import hashlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import ndimage
from surface_distance import metrics as surface_distance_metrics


SURFACE_DICE_TOLERANCE_MM = 1.0
LESION_CONNECTIVITY = 3  # scipy rank-3/connectivity-3 = 26-connected components

TARGET_DEFINITIONS = {
    "Non-enhancing tumor core": {1},
    "FLAIR hyperintensity / edema": {2},
    "Enhancing tissue": {3},
    "Resection cavity": {4},
    "Tumor-related region (1-3)": {1, 2, 3},
    "All postoperative regions (1-4)": {1, 2, 3, 4},
}

REQUIRED_PER_CASE_COLUMNS = [
    "run_id",
    "model",
    "cv_fold",
    "split_seed",
    "training_seed",
    "case_id",
    "patient_id",
    "target",
    "tp",
    "fp",
    "fn",
    "dice",
    "iou",
    "precision",
    "recall",
    "gt_present",
    "pred_present",
    "truth_voxels",
    "predicted_voxels",
    "voxel_spacing_x_mm",
    "voxel_spacing_y_mm",
    "voxel_spacing_z_mm",
    "surface_distance_defined",
    "hd95_mm",
    "average_surface_distance_mm",
    "average_surface_distance_gt_to_pred_mm",
    "average_surface_distance_pred_to_gt_mm",
    "surface_dice_1mm",
    "reference_lesions",
    "predicted_lesions",
    "detected_reference_lesions",
    "missed_reference_lesions",
    "false_positive_lesions",
    "lesion_sensitivity",
]

METRIC_DEFINITIONS = {
    "version": "mu_glioma_uniform_metrics_v2_fixed_reference_subset",
    "targets": {key: sorted(value) for key, value in TARGET_DEFINITIONS.items()},
    "overlap": {
        "dice": "2TP/(2TP+FP+FN)",
        "iou": "TP/(TP+FP+FN)",
        "precision": "TP/(TP+FP)",
        "recall": "TP/(TP+FN)",
        "both_reference_and_prediction_empty": "Dice/IoU/precision/recall are NaN and excluded from means",
        "reference_empty_prediction_present": "Dice/IoU/precision are 0; recall is NaN",
        "aggregation_subset": "all overlap summaries use the fixed reference-present scans only; reference-absent scans are reported separately",
    },
    "surface": {
        "implementation": "google-deepmind surface-distance 0.1 with physical NIfTI voxel spacing",
        "hd95_mm": "area-weighted symmetric robust Hausdorff distance at the 95th percentile",
        "average_surface_distance_mm": "unweighted mean of the two area-weighted directed average surface distances",
        "surface_dice_tolerance_mm": SURFACE_DICE_TOLERANCE_MM,
        "both_nonempty_required_for_distances": True,
        "one_empty_policy": "HD95/average distances NaN; surface Dice 0",
        "both_empty_policy": "all surface metrics NaN",
    },
    "lesion": {
        "connectivity": "26-connected components in 3D",
        "detection_rule": "a reference component is detected when at least one predicted voxel overlaps it",
        "false_positive_rule": "a predicted component is false-positive when it overlaps no reference voxel",
        "absent_reference_sensitivity": "NaN",
    },
    "absent_reference": {
        "false_positive_scan": "reference target absent and at least one target voxel predicted",
        "confidence_interval": "95% Wilson binomial interval across distinct scans",
        "independence_note": "seed repeats are not counted as new independent patients",
    },
}


def validate_discrete_label_volume(
    array: np.ndarray,
    context: str = "label volume",
    allowed_labels=(0, 1, 2, 3, 4),
) -> np.ndarray:
    """Validate before casting so overflow cannot hide invalid predictions."""
    array = np.asanyarray(array)
    assert np.issubdtype(array.dtype, np.number), (
        f"{context}: non-numeric dtype {array.dtype}"
    )
    assert np.all(np.isfinite(array)), f"{context}: NaN or infinite values"
    assert np.all(array == np.rint(array)), f"{context}: non-integer label values"
    observed = {int(value) for value in np.unique(array)}
    allowed = set(int(value) for value in allowed_labels)
    assert observed.issubset(allowed), (
        f"{context}: unexpected labels {sorted(observed - allowed)}"
    )
    return array.astype(np.uint8, copy=False)


def binary_overlap_metrics(truth: np.ndarray, prediction: np.ndarray) -> dict:
    truth = np.asarray(truth, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    tp = int(np.logical_and(prediction, truth).sum())
    fp = int(np.logical_and(prediction, ~truth).sum())
    fn = int(np.logical_and(~prediction, truth).sum())
    gt_present = bool(truth.any())
    pred_present = bool(prediction.any())
    if not gt_present and not pred_present:
        dice = iou = precision = recall = np.nan
    else:
        dice = (2 * tp) / max(2 * tp + fp + fn, 1)
        iou = tp / max(tp + fp + fn, 1)
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1) if gt_present else np.nan
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "gt_present": gt_present,
        "pred_present": pred_present,
        "truth_voxels": int(truth.sum()),
        "predicted_voxels": int(prediction.sum()),
    }


def surface_and_lesion_metrics(
    truth: np.ndarray,
    prediction: np.ndarray,
    spacing_mm,
) -> dict:
    truth = np.asarray(truth, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    spacing_mm = tuple(float(value) for value in spacing_mm)
    gt_present = bool(truth.any())
    pred_present = bool(prediction.any())

    if gt_present and pred_present:
        distances = surface_distance_metrics.compute_surface_distances(
            truth, prediction, spacing_mm=spacing_mm
        )
        directed_asd = surface_distance_metrics.compute_average_surface_distance(
            distances
        )
        hd95_mm = float(
            surface_distance_metrics.compute_robust_hausdorff(distances, 95.0)
        )
        asd_gt_to_pred = float(directed_asd[0])
        asd_pred_to_gt = float(directed_asd[1])
        average_surface_distance = (asd_gt_to_pred + asd_pred_to_gt) / 2.0
        surface_dice = float(
            surface_distance_metrics.compute_surface_dice_at_tolerance(
                distances, tolerance_mm=SURFACE_DICE_TOLERANCE_MM
            )
        )
        surface_distance_defined = True
    elif gt_present or pred_present:
        hd95_mm = np.nan
        asd_gt_to_pred = np.nan
        asd_pred_to_gt = np.nan
        average_surface_distance = np.nan
        surface_dice = 0.0
        surface_distance_defined = False
    else:
        hd95_mm = np.nan
        asd_gt_to_pred = np.nan
        asd_pred_to_gt = np.nan
        average_surface_distance = np.nan
        surface_dice = np.nan
        surface_distance_defined = False

    structure = ndimage.generate_binary_structure(3, LESION_CONNECTIVITY)
    truth_components, reference_lesions = ndimage.label(truth, structure=structure)
    prediction_components, predicted_lesions = ndimage.label(
        prediction, structure=structure
    )
    detected_reference_lesions = sum(
        bool(prediction[truth_components == component].any())
        for component in range(1, reference_lesions + 1)
    )
    false_positive_lesions = sum(
        not bool(truth[prediction_components == component].any())
        for component in range(1, predicted_lesions + 1)
    )
    missed_reference_lesions = reference_lesions - detected_reference_lesions
    lesion_sensitivity = (
        detected_reference_lesions / reference_lesions
        if reference_lesions
        else np.nan
    )
    return {
        "voxel_spacing_x_mm": spacing_mm[0],
        "voxel_spacing_y_mm": spacing_mm[1],
        "voxel_spacing_z_mm": spacing_mm[2],
        "surface_distance_defined": surface_distance_defined,
        "hd95_mm": hd95_mm,
        "average_surface_distance_mm": average_surface_distance,
        "average_surface_distance_gt_to_pred_mm": asd_gt_to_pred,
        "average_surface_distance_pred_to_gt_mm": asd_pred_to_gt,
        "surface_dice_1mm": surface_dice,
        "reference_lesions": int(reference_lesions),
        "predicted_lesions": int(predicted_lesions),
        "detected_reference_lesions": int(detected_reference_lesions),
        "missed_reference_lesions": int(missed_reference_lesions),
        "false_positive_lesions": int(false_positive_lesions),
        "lesion_sensitivity": lesion_sensitivity,
    }


def evaluate_case_targets(
    truth_labels: np.ndarray,
    prediction_labels: np.ndarray,
    spacing_mm,
    identity: dict,
) -> list[dict]:
    rows = []
    for target_name, target_labels in TARGET_DEFINITIONS.items():
        truth = np.isin(truth_labels, list(target_labels))
        prediction = np.isin(prediction_labels, list(target_labels))
        row = dict(identity)
        row["target"] = target_name
        row.update(binary_overlap_metrics(truth, prediction))
        row.update(surface_and_lesion_metrics(truth, prediction, spacing_mm))
        rows.append(row)
    return rows


def metrics_from_counts(tp: int, fp: int, fn: int):
    if tp == 0 and fp == 0 and fn == 0:
        return np.nan, np.nan, np.nan, np.nan
    dice = (2 * tp) / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1) if (tp + fn) > 0 else np.nan
    return dice, iou, precision, recall


def wilson_interval(successes: int, total: int, z: float = 1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    rate = successes / total
    denominator = 1 + z * z / total
    center = (rate + z * z / (2 * total)) / denominator
    margin = z * math.sqrt(
        rate * (1 - rate) / total + z * z / (4 * total * total)
    ) / denominator
    return max(0.0, center - margin), min(1.0, center + margin)


def build_uniform_summaries(scores: pd.DataFrame):
    missing = [column for column in REQUIRED_PER_CASE_COLUMNS if column not in scores]
    assert not missing, f"Missing required per-case columns: {missing}"

    patient_rows = []
    for (patient_id, target), group in scores.groupby(
        ["patient_id", "target"], sort=True
    ):
        present = group[group.gt_present]
        absent = group[~group.gt_present]
        tp, fp, fn = (
            int(present.tp.sum()),
            int(present.fp.sum()),
            int(present.fn.sum()),
        )
        dice, iou, precision, recall = metrics_from_counts(tp, fp, fn)
        reference_lesions = int(present.reference_lesions.sum())
        detected_lesions = int(present.detected_reference_lesions.sum())
        patient_rows.append(
            {
                "run_id": group.run_id.iloc[0],
                "model": group.model.iloc[0],
                "cv_fold": int(group.cv_fold.iloc[0]),
                "split_seed": int(group.split_seed.iloc[0]),
                "training_seed": int(group.training_seed.iloc[0]),
                "patient_id": patient_id,
                "target": target,
                "scan_count": int(group.case_id.nunique()),
                "reference_present_scan_count": int(present.case_id.nunique()),
                "reference_absent_scan_count": int(absent.case_id.nunique()),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "pooled_dice": dice,
                "pooled_iou": iou,
                "pooled_precision": precision,
                "pooled_recall": recall,
                "mean_scan_dice": float(present.dice.mean()),
                "median_scan_hd95_mm": float(present.hd95_mm.median()),
                "mean_scan_average_surface_distance_mm": float(
                    present.average_surface_distance_mm.mean()
                ),
                "mean_scan_surface_dice_1mm": float(
                    present.surface_dice_1mm.mean()
                ),
                "reference_lesions": reference_lesions,
                "detected_reference_lesions": detected_lesions,
                "missed_reference_lesions": int(
                    present.missed_reference_lesions.sum()
                ),
                "false_positive_lesions_on_reference_present_scans": int(
                    present.false_positive_lesions.sum()
                ),
                "false_positive_scans_when_reference_absent": int(
                    absent.pred_present.sum()
                ),
                "false_positive_lesions_when_reference_absent": int(
                    absent.false_positive_lesions.sum()
                ),
                "lesion_sensitivity": (
                    detected_lesions / reference_lesions
                    if reference_lesions
                    else np.nan
                ),
            }
        )
    patient_scores = pd.DataFrame(patient_rows)

    summary_rows = []
    absent_rows = []
    for target_name in TARGET_DEFINITIONS:
        subset = scores[scores.target == target_name]
        present = subset[subset.gt_present]
        dice_values = present.dice.dropna()
        q1, q3 = dice_values.quantile([0.25, 0.75])
        reference_lesions = int(present.reference_lesions.sum())
        detected_lesions = int(present.detected_reference_lesions.sum())
        absent = subset[~subset.gt_present]
        false_positive_scans = int(absent.pred_present.sum())
        ci_low, ci_high = wilson_interval(false_positive_scans, len(absent))
        summary_rows.append(
            {
                "run_id": subset.run_id.iloc[0],
                "model": subset.model.iloc[0],
                "cv_fold": int(subset.cv_fold.iloc[0]),
                "split_seed": int(subset.split_seed.iloc[0]),
                "training_seed": int(subset.training_seed.iloc[0]),
                "target": target_name,
                "mean_dice": float(dice_values.mean()),
                "sd_dice": float(dice_values.std(ddof=1)),
                "median_dice": float(dice_values.median()),
                "q1_dice": float(q1),
                "q3_dice": float(q3),
                "mean_iou": float(present.iou.mean()),
                "mean_precision": float(present.precision.mean()),
                "mean_recall": float(present.recall.mean()),
                "median_hd95_mm": float(present.hd95_mm.median()),
                "mean_average_surface_distance_mm": float(
                    present.average_surface_distance_mm.mean()
                ),
                "mean_surface_dice_1mm": float(
                    present.surface_dice_1mm.mean()
                ),
                "reference_lesions": reference_lesions,
                "detected_reference_lesions": detected_lesions,
                "missed_reference_lesions": int(
                    present.missed_reference_lesions.sum()
                ),
                "false_positive_lesions_on_reference_present_scans": int(
                    present.false_positive_lesions.sum()
                ),
                "lesion_sensitivity": (
                    detected_lesions / reference_lesions
                    if reference_lesions
                    else np.nan
                ),
                "validation_scans": int(subset.case_id.nunique()),
                "validation_patients": int(subset.patient_id.nunique()),
                "reference_present_scans": int(present.case_id.nunique()),
                "reference_present_patients": int(present.patient_id.nunique()),
                "absent_reference_scans": int(len(absent)),
                "absent_reference_patients": int(absent.patient_id.nunique()),
                "absent_reference_false_positive_scans": false_positive_scans,
                "absent_reference_false_positive_rate": (
                    false_positive_scans / len(absent) if len(absent) else np.nan
                ),
                "absent_reference_false_positive_rate_ci95_low": ci_low,
                "absent_reference_false_positive_rate_ci95_high": ci_high,
            }
        )
        absent_rows.append(
            {
                "run_id": subset.run_id.iloc[0],
                "model": subset.model.iloc[0],
                "cv_fold": int(subset.cv_fold.iloc[0]),
                "training_seed": int(subset.training_seed.iloc[0]),
                "target": target_name,
                "absent_reference_scans": int(len(absent)),
                "absent_reference_patients": int(absent.patient_id.nunique()),
                "false_positive_scans": false_positive_scans,
                "false_positive_scan_rate": (
                    false_positive_scans / len(absent) if len(absent) else np.nan
                ),
                "false_positive_scan_rate_ci95_low": ci_low,
                "false_positive_scan_rate_ci95_high": ci_high,
                "mean_predicted_voxels_when_absent": float(
                    absent.predicted_voxels.mean()
                ),
                "median_predicted_voxels_when_absent": float(
                    absent.predicted_voxels.median()
                ),
                "false_positive_lesions_when_absent": int(
                    absent.false_positive_lesions.sum()
                ),
            }
        )
    return patient_scores, pd.DataFrame(summary_rows), pd.DataFrame(absent_rows)


def sha256_file(path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_prediction_manifest(prediction_dir, validation_cases: list[dict]):
    case_lookup = {case["id"]: case for case in validation_cases}
    rows = []
    for path in sorted(Path(prediction_dir).glob("*.nii.gz")):
        case_id = path.name.removesuffix(".nii.gz")
        case = case_lookup.get(case_id)
        rows.append(
            {
                "case_id": case_id,
                "patient_id": case["patient"] if case else "",
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
    return pd.DataFrame(rows)


prediction_dir = OUTPUT / "outer_test_predictions"
if prediction_dir.exists():
    # Inference is intentionally restarted as one atomic phase. This prevents
    # partial or stale prediction sets from an interrupted attempt being mixed
    # with the frozen best model's new outputs.
    shutil.rmtree(prediction_dir)
prediction_dir.mkdir(parents=True, exist_ok=True)


def window_starts(size, patch, stride):
    if size <= patch:
        return [0]
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def predict_volume(image):
    original_shape = image.shape[:3]
    pads = [(0, max(0, patch - size)) for size, patch in zip(original_shape, PATCH_SIZE)]
    if any(after for _, after in pads):
        image = np.pad(image, pads + [(0, 0)])

    probability_sum = np.zeros(image.shape[:3] + (NUM_CLASSES,), dtype=np.float32)
    overlap_count = np.zeros(image.shape[:3], dtype=np.float32)
    axis_starts = [
        window_starts(size, patch, stride)
        for size, patch, stride in zip(image.shape[:3], PATCH_SIZE, INFERENCE_STRIDE)
    ]
    for x0 in axis_starts[0]:
        for y0 in axis_starts[1]:
            for z0 in axis_starts[2]:
                patch = image[
                    x0 : x0 + PATCH_SIZE[0],
                    y0 : y0 + PATCH_SIZE[1],
                    z0 : z0 + PATCH_SIZE[2],
                ]
                probabilities = model(patch[None], training=False).numpy()[0]
                probability_sum[
                    x0 : x0 + PATCH_SIZE[0],
                    y0 : y0 + PATCH_SIZE[1],
                    z0 : z0 + PATCH_SIZE[2],
                ] += probabilities
                overlap_count[
                    x0 : x0 + PATCH_SIZE[0],
                    y0 : y0 + PATCH_SIZE[1],
                    z0 : z0 + PATCH_SIZE[2],
                ] += 1
    probability = probability_sum / np.maximum(overlap_count[..., None], 1)
    return probability[: original_shape[0], : original_shape[1], : original_shape[2]]


rows = []
inference_timing_rows = []
started = time.perf_counter()
for number, case in enumerate(outer_test_cases, 1):
    case_started = time.perf_counter()
    image, truth_4d = load_case(case)
    truth_labels = truth_4d[..., 0].astype(np.uint8)
    probabilities = predict_volume(image)
    prediction_labels = np.argmax(probabilities, axis=-1).astype(np.uint8)
    reference = nib.load(case["mask"])
    spacing_mm = tuple(float(value) for value in reference.header.get_zooms()[:3])
    rows.extend(
        evaluate_case_targets(
            truth_labels,
            prediction_labels,
            spacing_mm,
            {
                "run_id": RUN_ID,
                "model": MODEL_DISPLAY_NAME,
                "cv_fold": CV_FOLD,
                "split_seed": SPLIT_SEED,
                "training_seed": TRAINING_SEED,
                "case_id": case["id"],
                "patient_id": case["patient"],
            },
        )
    )

    header = reference.header.copy()
    header.set_data_dtype(np.uint8)
    nib.save(
        nib.Nifti1Image(prediction_labels, reference.affine, header),
        prediction_dir / f"{case['id']}.nii.gz",
    )
    inference_timing_rows.append(
        {
            "run_id": RUN_ID,
            "case_id": case["id"],
            "patient_id": case["patient"],
            "wall_seconds_including_io": time.perf_counter() - case_started,
        }
    )
    del image, truth_4d, probabilities, prediction_labels
    if number % 5 == 0 or number == len(outer_test_cases):
        print(
            f"Evaluated {number}/{len(outer_test_cases)} outer-test scans; "
            f"elapsed {(time.perf_counter() - started) / 60:.1f} min"
        )

scores = pd.DataFrame(rows)
assert list(scores.columns) == REQUIRED_PER_CASE_COLUMNS
assert scores["case_id"].nunique() == EXPECTED_OUTER_TEST_CASES
assert scores["patient_id"].nunique() == EXPECTED_OUTER_TEST_PATIENTS
scores.to_csv(OUTPUT / "outer_test_per_case_metrics.csv", index=False)

prediction_manifest = build_prediction_manifest(prediction_dir, outer_test_cases)
assert len(prediction_manifest) == EXPECTED_OUTER_TEST_CASES
assert prediction_manifest.case_id.nunique() == EXPECTED_OUTER_TEST_CASES
prediction_manifest.to_csv(OUTPUT / "outer_test_prediction_manifest.csv", index=False)
inference_timing = pd.DataFrame(inference_timing_rows)
inference_timing.to_csv(OUTPUT / "outer_test_inference_timing.csv", index=False)
inference_seconds = float(inference_timing.wall_seconds_including_io.sum())
efficiency = json.loads((OUTPUT / "efficiency.json").read_text())
efficiency.update(
    {
        "outer_test_inference_total_wall_seconds_including_io": inference_seconds,
        "outer_test_cases": int(len(inference_timing)),
        "outer_test_inference_mean_seconds_per_case": float(
            inference_timing.wall_seconds_including_io.mean()
        ),
        "outer_test_inference_median_seconds_per_case": float(
            inference_timing.wall_seconds_including_io.median()
        ),
        "outer_test_cases_per_hour": (
            3600 * len(inference_timing) / inference_seconds
            if inference_seconds > 0
            else None
        ),
    }
)
(OUTPUT / "efficiency.json").write_text(
    json.dumps(efficiency, indent=2), encoding="utf-8"
)
(OUTPUT / "metric_definitions.json").write_text(
    json.dumps(METRIC_DEFINITIONS, indent=2), encoding="utf-8"
)
print("Saved detailed per-case metrics:", scores.shape)
print("Indexed untouched outer-test prediction volumes:", len(prediction_manifest))


## 10. Per-patient metrics, final summary, and training curves


In [ ]:

patient_scores, summary, absent_reference_summary = build_uniform_summaries(scores)
patient_scores.to_csv(OUTPUT / "outer_test_per_patient_metrics.csv", index=False)
summary.to_csv(OUTPUT / "final_summary.csv", index=False)
absent_reference_summary.to_csv(
    OUTPUT / "absent_reference_false_positive_summary.csv", index=False
)
display(summary)
display(absent_reference_summary)

training_log = pd.read_csv(TRAINING_LOG)
best_index = training_log["val_mean_foreground_dice"].idxmax()
best_epoch = int(training_log.loc[best_index, "epoch"]) + 1
completed_epochs = int(training_log.epoch.max()) + 1
best_patch_dice = float(training_log.loc[best_index, "val_mean_foreground_dice"])
stop_reason = "early_stopping" if completed_epochs < EPOCHS else "maximum_epochs"

run_summary = {
    "archive_contract_version": "mu_glioma_detailed_zip_v2_nested",
    "run_id": RUN_ID,
    "cv_fold": CV_FOLD,
    "split_seed": SPLIT_SEED,
    "training_seed": TRAINING_SEED,
    "model": MODEL_DISPLAY_NAME,
    "architecture_id": ARCHITECTURE_ID,
    "status": "complete",
    "split_sha256": split_sha256,
    "maximum_epochs": EPOCHS,
    "completed_epochs": completed_epochs,
    "stop_reason": stop_reason,
    "best_epoch": best_epoch,
    "best_patch_validation_mean_foreground_dice": best_patch_dice,
    "best_weights_restored": True,
    "checkpoint_selection_role": "patient_grouped_inner_tuning",
    "final_evaluation_role": "untouched_outer_internal_test",
    "fit_train_scans": EXPECTED_FIT_TRAIN_CASES,
    "fit_train_patients": EXPECTED_FIT_TRAIN_PATIENTS,
    "tuning_scans": EXPECTED_TUNING_CASES,
    "tuning_patients": EXPECTED_TUNING_PATIENTS,
    "outer_test_scans": int(scores.case_id.nunique()),
    "outer_test_patients": int(scores.patient_id.nunique()),
    "prediction_files": len(prediction_manifest),
    "surface_dice_tolerance_mm": SURFACE_DICE_TOLERANCE_MM,
    "lesion_connectivity": "26-connected",
    "targets": summary.to_dict(orient="records"),
}
assert run_summary["prediction_files"] == EXPECTED_OUTER_TEST_CASES
(OUTPUT / "final_summary.json").write_text(
    json.dumps(run_summary, indent=2), encoding="utf-8"
)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(training_log.epoch + 1, training_log.loss, label="training")
axes[0].plot(training_log.epoch + 1, training_log.val_loss, label="inner tuning")
axes[0].set(title="Categorical CE + Dice loss", xlabel="Epoch")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(
    training_log.epoch + 1,
    training_log.mean_foreground_dice,
    label="training",
)
axes[1].plot(
    training_log.epoch + 1,
    training_log.val_mean_foreground_dice,
    label="inner tuning",
)
axes[1].axvline(best_epoch, color="black", linestyle="--", label=f"best={best_epoch}")
axes[1].set(title="Inner-tuning patch foreground Dice", xlabel="Epoch")
axes[1].grid(alpha=0.3)
axes[1].legend()
figure.tight_layout()
figure.savefig(OUTPUT / "learning_curves.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Best epoch: {best_epoch}; inner-tuning patch Dice: {best_patch_dice:.4f}")
print("Completed epochs:", completed_epochs, "/ stop reason:", stop_reason)


## 11. Package the model, predictions, metrics, split, and configuration


In [ ]:

required_files = [
    OUTPUT / "patient_split.json",
    OUTPUT / "model_config.json",
    OUTPUT / "environment.json",
    OUTPUT / "metric_definitions.json",
    OUTPUT / "fit_complete.json",
    OUTPUT / "training_log.csv",
    OUTPUT / "efficiency.json",
    OUTPUT / "outer_test_per_case_metrics.csv",
    OUTPUT / "outer_test_per_patient_metrics.csv",
    OUTPUT / "outer_test_inference_timing.csv",
    OUTPUT / "absent_reference_false_positive_summary.csv",
    OUTPUT / "final_summary.csv",
    OUTPUT / "final_summary.json",
    OUTPUT / "outer_test_prediction_manifest.csv",
    OUTPUT / "learning_curves.png",
    BEST_MODEL,
]
missing_required = [str(path) for path in required_files if not path.exists()]
assert not missing_required, f"Required artifacts missing: {missing_required}"
assert len(list(prediction_dir.glob("*.nii.gz"))) == EXPECTED_OUTER_TEST_CASES

manifest = {
    "archive_contract_version": "mu_glioma_detailed_zip_v2_nested",
    "bundle_name": f"{OUTPUT_NAME}_results.zip",
    "run_id": RUN_ID,
    "model": MODEL_DISPLAY_NAME,
    "cv_fold": CV_FOLD,
    "split_seed": SPLIT_SEED,
    "training_seed": TRAINING_SEED,
    "split_sha256": split_sha256,
    "checkpoint_selection_role": "patient_grouped_inner_tuning",
    "final_evaluation_role": "untouched_outer_internal_test",
    "prediction_files": len(list(prediction_dir.glob("*.nii.gz"))),
    "per_case_metric_rows": int(len(scores)),
    "required_per_case_columns_present": all(
        column in scores.columns for column in REQUIRED_PER_CASE_COLUMNS
    ),
    "includes_all_prediction_volumes": True,
    "includes_best_model_or_checkpoint": BEST_MODEL.exists(),
    "includes_surface_metrics": True,
    "includes_lesion_metrics": True,
    "includes_absent_reference_uncertainty": True,
}
(OUTPUT / "artifact_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

# Build this only after the manifest exists, so every other returned artifact
# is covered. Remove a prior index first when a completed notebook is rerun.
artifact_checksums_path = OUTPUT / "artifact_checksums.csv"
if artifact_checksums_path.exists():
    artifact_checksums_path.unlink()
checksum_rows = []
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and "training_backup" not in path.parts:
        checksum_rows.append(
            {
                "relative_path": str(path.relative_to(OUTPUT)),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
artifact_checksums = pd.DataFrame(checksum_rows)
artifact_checksums.to_csv(artifact_checksums_path, index=False)

bundle_path = COLAB_OUTPUT_ROOT / f"{OUTPUT_NAME}_results.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_STORED) as bundle:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "training_backup" not in path.parts:
            bundle.write(path, arcname=str(Path(OUTPUT_NAME) / path.relative_to(OUTPUT)))

print(json.dumps(manifest, indent=2))
print(f"Bundle size: {bundle_path.stat().st_size / (1024 ** 3):.2f} GiB")
print("Return this complete Drive archive:", bundle_path)


## Completion checklist

The Colab run is complete only when the notebook reports 119
prediction files from 41 untouched outer-test patients
and creates `MU_Glioma_no3_unet_fold3_seed2026_results.zip` under `MyDrive/MU_Glioma_35_Run_Outputs` with
archive contract `mu_glioma_detailed_zip_v2_nested`. Return that ZIP with run
ID `no3`; an interrupted Drive checkpoint or output directory does not
count as a result.
